<a href="https://colab.research.google.com/github/Squad-Nina-da-Hora/wmc-projeto-final-lifestyle/blob/main/analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# **Análise de Sono e Estilo de Vida**
---


🎯 **Objetivo:**


---


Projeto Final

Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## 1. Configurações Iniciais

Variáveis da base de dados:

- `Person ID`: Identificador único de cada indivíduo. 
- `Gender`: Gênero da pessoa (Masculino/Feminino). 
- `Age`: Idade da pessoa em anos. 
- `Occupation`: Ocupação ou profissão da pessoa. 
- `Sleep Duration` (hours): Número de horas que a pessoa dorme por dia. 
- `Quality of Sleep` (scale: 1-10): Avaliação subjetiva da qualidade do sono, variando de 1 a 10. 
- `Physical Activity Level` (minutes/day): Número de minutos de atividade física diária. 
- `Stress Level` (scale: 1-10): Avaliação subjetiva do nível de estresse, variando de 1 a 10. 
- `BMI Category`: Categoria de IMC (por exemplo: Abaixo do peso, Normal, Sobrepeso). 
- `Blood Pressure` (systolic/diastolic): Medida da pressão arterial, indicada como pressão sistólica sobre diastólica. 
- `Heart Rate` (bpm): Frequência cardíaca em repouso, em batimentos por minuto. 
- `Daily Steps`: Número de passos dados por dia. 
- `Sleep Disorder`: Presença ou ausência de distúrbio do sono (Nenhum, Insomnia, Sleep Apnea). 

In [ ]:
# ==============================
# IMPORTACOES
# ==============================

import kagglehub    # Para baixar datasets do Kaggle
import numpy as np  # Para operacoes numericas e arrays
import pandas as pd  # Para manipulaco e analise de data frames
#from IPython.display import display, Markdown
# Bibliotecas para criacao de graficos
import matplotlib.pyplot as plt
import seaborn as sns
"""
import missingno as msno
# Bibliotecas para criacao de modelos de ML
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
"""

# Carregamento da base de dados
url = f'{kagglehub.dataset_download("uom190346a/sleep-health-and-lifestyle-dataset")}/Sleep_health_and_lifestyle_dataset.csv'
df = pd.read_csv(url).set_index('Person ID')

In [ ]:
# ==============================
# VARIAVEIS PARA REUTILIZACAO
# ==============================

ALVO = 'Quality of Sleep' # Constante da varivel alvo do modelo
ALVO_CAT = 'sono_classificacao' # Contante da variavel alvo categorizada

# Configuracoes visuais dos graficos

PALETA = 'flare'

##  2. Análise exploratória dos dados

In [ ]:
df.info()

In [ ]:
print('='*50)
print('Describe das colunas numericas')
print('='*50)
display(df.describe())

print('='*50)
print('Describe das colunas categoricas')
print('='*50)
df.describe(include=['category', 'object', 'string'])

In [ ]:
# Checa os valores unicos incomuns das colunas categoricas
for col in df.select_dtypes(include=['category', 'object', 'string']).columns:
  print(f"\nColuna: {col}")
  print(df[col].unique())

In [ ]:
# Padroniza categorias iguais com nomes diferentes em 'BMI Category'
df['BMI Category'] = df['BMI Category'].str.replace('Normal Weight', 'Normal', regex=False)

# Converte colunas categorias para o tipo 'category'
cols_categoricas = ['Gender', 'Occupation', 'BMI Category', 'Sleep Disorder']
df[cols_categoricas] = df[cols_categoricas].astype('category')

In [ ]:
# Separa e converte a pressao arterial em sistolica e diastolica (int) e calcula a PAM
pressoes_separadas = df['Blood Pressure'].str.split('/', expand=True).astype(int)
sistolica = pressoes_separadas[0]
diastolica = pressoes_separadas[1]
df['PAM'] = (sistolica + 2 * diastolica) / 3

In [ ]:
# Cria a coluna com 3 classificacoes, espacadas uniformemente entre a idade minima e maxima
idade_rotulos = ['Mais jovem', 'Intermediária', 'Mais velha']
idade_bins = np.linspace(df['Age'].min(), df['Age'].max(), 4)

df['idade_classificacao'] = pd.cut(df['Age'], bins=idade_bins, labels=idade_rotulos, include_lowest=True)

In [ ]:
# Cria a coluna de classificacao da qualidade de sono (var alvo)
sono_rotulos = ['Ruim', 'Moderada', 'Boa']
sono_bins = [0, 4, 6, 10]

df[ALVO_CAT] = pd.cut(df[ALVO], bins=sono_bins, labels=sono_rotulos, include_lowest=True)

In [ ]:
# Analisa distribuicao e desbalanceamento da variavel alvo
n_alvo_cat = df[ALVO_CAT].nunique() # quantidade de categorias na variavel alvo
alvo_cores = sns.color_palette(PALETA, n_colors=n_alvo_cat)

# Tabela
display((df[ALVO_CAT].value_counts(normalize=True) * 100).rename('Percentual'))

# Grafico
plt.figure(figsize=(8, 6))
ax = sns.countplot(data=df, x=ALVO_CAT, palette=alvo_cores, hue=ALVO_CAT, legend=False)
plt.title(f'Distribuição da Variável Alvo `{ALVO_CAT}`', fontsize=12, fontweight='bold')
plt.xlabel(f'{ALVO_CAT}')
plt.ylabel('Contagem')

# Adiciona porcentagens nas barras
total = len(df)
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{(height/total)*100:.1f}%',
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom', xytext=(0, 3),
                textcoords='offset points')
plt.tight_layout()
plt.show()

In [ ]:
# Remove colunas que nao serao utilizadas na analise
df = df.drop(['Blood Pressure'], axis=1)

In [ ]:
print('='*50)
print('Diagnóstico de Tipagem e Qualidade')
print('='*50)

nulos_str = (df.isnull().sum().astype(str)
  + ' ('
  + (df.isnull().mean() * 100).map('{:.1f}%'.format)
  + ')')

pd.DataFrame({
  'Tipo': df.dtypes.astype(str),
  'Valores Nulos': nulos_str,
  'Valores Únicos': df.nunique(),
}).sort_values('Valores Nulos', ascending=False)

## 3. Modelo de ML

### 3.1. Avaliação dos Modelos

## 4. Cálculo do gap de Recall entre as classes

## 5. Validação cruzada e generalização

## 6. Conclusão Geral